# PRM800K — Test 1 with Microsoft Phi-3-mini-4k-instruct

**Goal:** Replicate Test 1 from the Gemma-2B experiment but using `microsoft/Phi-3-mini-4k-instruct`.

**What we are testing:**
Can Phi-3's internal hidden states (its internal understanding of text) distinguish
correct reasoning steps from incorrect ones, when trained on real PRM800K human labels?

**What we are NOT doing here:**
We skip GSM8K entirely — this notebook only runs Test 1 (PRM classifier accuracy).

**Success threshold:**
- `> 80%` → GO — Phi-3 hidden states encode step quality well
- `70–80%` → Weak signal
- `< 70%` → NO-GO (Gemma-2B scored 66.3% here)

## Cell 1 — Hugging Face Authentication
Phi-3 is a gated model. Store your HF token as a Colab secret named `HF_TOKEN`
via the 🔑 key icon in the left sidebar.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN'))
print('✅ Hugging Face login successful.')

✅ Hugging Face login successful.


## Cell 2 — Install Dependencies
`git-lfs` is required to download the large PRM800K `.jsonl` files.
`flash-attn` is optional but speeds up Phi-3 significantly on T4.

In [ ]:
# !apt-get install -y git-lfs -q
# !pip install -q transformers accelerate scikit-learn tqdm
print('✅ Dependencies ready.')

✅ Dependencies ready.


## Cell 3 — Clone OpenAI PRM800K Repository
Downloads the real human-annotated step labels.
Only clones once — skips if folder already exists.

In [ ]:
import os

REPO_DIR = '/content/prm800k'
DATA_DIR = f'{REPO_DIR}/prm800k/data'

if os.path.exists(DATA_DIR):
    print('✅ PRM800K repo already cloned — skipping.')
    print(f'   Files: {os.listdir(DATA_DIR)}')
else:
    print('Cloning PRM800K (this may take a few minutes)...')
    !git lfs install
    !git clone https://github.com/openai/prm800k.git {REPO_DIR}
    print('✅ PRM800K cloned successfully.')
    print(f'   Files: {os.listdir(DATA_DIR)}')

Cloning PRM800K (this may take a few minutes)...
Git LFS initialized.
Cloning into '/content/prm800k'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 28 (delta 1), reused 0 (delta 0), pack-reused 20 (from 1)
Receiving objects: 100% (28/28), 4.86 MiB | 9.71 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Filtering content: 100% (6/6), 465.82 MiB | 8.72 MiB/s, done.
✅ PRM800K cloned successfully.
   Files: ['phase1_test.jsonl', 'phase2_train.jsonl', 'phase1_train.jsonl', 'phase2_test.jsonl']


## Cell 4 — Load Phi-3-mini-4k-instruct Model & Tokenizer

Key differences from Gemma-2B:
- Phi-3-mini has **3.8B parameters** vs Gemma's 2B — more powerful but slightly slower
- Uses `trust_remote_code=True` because Phi-3 has custom architecture code
- We load in `float16` to fit on T4 GPU

**Guard:** Skips loading if model is already in memory.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
# MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'Device: {DEVICE}')
print(f'Model : {MODEL_ID}')

if 'model' in dir() and model is not None:
    print('✅ Model already loaded — skipping.')
else:
    print('Loading Phi-3 tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True
    )

    print('Loading Phi-3 model weights (~7GB download, ~5 min on T4)...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,   # half precision to fit in GPU memory
        device_map='auto',           # automatically places layers on GPU/CPU
        output_hidden_states=True,    # we NEED hidden states for embeddings
    )
    model.eval()  # inference mode — disables dropout etc.
    print(f'✅ Phi-3 loaded successfully on {DEVICE}.')

Device: cuda
Model : mistralai/Mistral-7B-Instruct-v0.3
Loading Phi-3 tokenizer...
Loading Phi-3 model weights (~7GB download, ~5 min on T4)...


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Phi-3 loaded successfully on cuda.


## Cell 5 — Load Real PRM800K Step-Level Labels

Reads the `.jsonl` files and extracts:
- The text of each reasoning step
- The human label: **correct (1)** or **incorrect (0)**

Label mapping from PRM800K:
- `+1` → step is correct → label **1**
- `-1` → step is wrong  → label **0**
- `0`  → step is neutral/skipped → **ignored**

In [ ]:
import json
import os

if 'prm_texts' in dir() and len(prm_texts) > 0:
    print(f'✅ PRM800K labels already loaded — {len(prm_texts)} steps.')
else:
    DATA_DIR = '/content/prm800k/prm800k/data'
    prm_texts  = []   # step text
    prm_labels = []   # 1 = correct, 0 = wrong

    for split in ['train', 'test']:
        fpath = os.path.join(DATA_DIR, f'phase2_{split}.jsonl')
        if not os.path.exists(fpath):
            print(f'⚠️  File not found: {fpath} — skipping.')
            continue

        with open(fpath) as f:
            for line in f:
                rec = json.loads(line)
                question = rec.get('question', {}).get('problem', '')
                for step in rec.get('label', {}).get('steps', []):
                    chosen = step.get('chosen_completion')
                    completions = step.get('completions', [])
                    if chosen is None or chosen >= len(completions):
                        continue
                    comp = completions[chosen]
                    rating = comp.get('rating')       # +1, -1, or 0
                    text   = comp.get('text', '').strip()
                    if not text or rating == 0:
                        continue
                    label = 1 if rating == 1 else 0   # +1 → correct, -1 → wrong
                    prm_texts.append(question + ' | ' + text)
                    prm_labels.append(label)

    correct_count = sum(prm_labels)
    wrong_count   = len(prm_labels) - correct_count
    print(f'✅ Loaded {len(prm_texts):,} labeled steps from PRM800K.')
    print(f'   Correct steps : {correct_count:,}')
    print(f'   Wrong steps   : {wrong_count:,}')

✅ Loaded 579,334 labeled steps from PRM800K.
   Correct steps : 543,304
   Wrong steps   : 36,030


## Cell 6 — Embed PRM800K Steps Using Phi-3 Hidden States

For each reasoning step, Phi-3 reads the text and produces a hidden state —
a vector of numbers representing its internal understanding of that step.

We take the **last hidden layer** and **mean-pool** across all tokens
to get one fixed-size vector per step.

Results are cached to disk — safe to re-run without re-embedding.

**Note:** Phi-3-mini has a hidden size of **3072** (vs Gemma-2B's 2048),
so the embeddings are richer.

In [ ]:
import numpy as np
import os
from tqdm import tqdm

tokenizer.pad_token = tokenizer.eos_token

EMB_CACHE  = '/content/phi3_prm_embeddings.npz'
MAX_EMBED  = 2000   # number of PRM800K steps to embed (balanced: 2500 correct + 2500 wrong)
MAX_LENGTH = 128    # token truncation length

if os.path.exists(EMB_CACHE):
    print('✅ Loading embeddings from cache...')
    data       = np.load(EMB_CACHE)
    embeddings = data['embeddings']
    labels     = data['labels']
    print(f'   Loaded {len(embeddings):,} embeddings from {EMB_CACHE}')
else:
    # ── Balance the dataset: equal correct and wrong steps ────────────────
    import random
    correct_idx = [i for i, l in enumerate(prm_labels) if l == 1]
    wrong_idx   = [i for i, l in enumerate(prm_labels) if l == 0]
    n_each      = min(MAX_EMBED // 2, len(correct_idx), len(wrong_idx))

    random.seed(42)
    selected_idx = random.sample(correct_idx, n_each) + random.sample(wrong_idx, n_each)
    random.shuffle(selected_idx)

    sel_texts  = [prm_texts[i]  for i in selected_idx]
    sel_labels = [prm_labels[i] for i in selected_idx]

    print(f'Embedding {len(sel_texts):,} steps ({n_each} correct + {n_each} wrong)...')
    print('This takes ~10-15 min on T4. Go grab a coffee ☕')

    embeddings = []

    for text in tqdm(sel_texts, desc='Embedding'):
        inputs = tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        # Last hidden layer, mean-pooled across tokens
        hidden = outputs.hidden_states[-1]                        # shape: (1, seq_len, hidden_size)
        mask   = inputs['attention_mask'].unsqueeze(-1).float()   # shape: (1, seq_len, 1)
        emb    = (hidden * mask).sum(1) / mask.sum(1)             # mean pool → (1, hidden_size)
        embeddings.append(emb.squeeze(0).cpu().float().numpy())

    embeddings = np.array(embeddings)
    labels     = np.array(sel_labels)



In [ ]:
np.savez(EMB_CACHE, embeddings=embeddings, labels=labels)
print(f'\n✅ Embeddings saved to {EMB_CACHE}')
print(f'   Shape: {embeddings.shape}  (steps × hidden_size)')

## Cell 7 — Train PRM Classifier on Phi-3 Embeddings

Trains a Logistic Regression classifier on the embeddings.
- 80% of data used for training
- 20% held out for testing (never seen during training)
- Accuracy is measured on the test split only

This is the **same classifier setup** used for Gemma-2B so results are directly comparable.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# ── Train / test split ────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels   # ensures balanced split
)

print(f'Training samples : {len(X_train):,}')
print(f'Test samples     : {len(X_test):,}')
print('Training classifier...')

clf = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0
)
clf.fit(X_train, y_train)

# ── Evaluate ─────────────────────────────────────────────────────
y_pred       = clf.predict(X_test)
prm_accuracy = accuracy_score(y_test, y_pred)

print(f'\n✅ PRM Classifier trained.')
print(f'   Test accuracy : {prm_accuracy:.1%}')
print()
print('Full classification report:')
print(classification_report(y_test, y_pred, target_names=['Wrong', 'Correct']))

## Cell 8 — GO / NO-GO Verdict

Final result compared against the same thresholds used in the Gemma-2B experiment,
so you can directly compare both models.

In [ ]:
print('=' * 60)
print('  PRM800K TEST 1 — PHI-3-MINI vs GEMMA-2B COMPARISON')
print('=' * 60)
print()
print('HYPOTHESIS:')
print('  Phi-3-mini hidden states can classify PRM800K steps')
print('  as correct/incorrect using human-annotated labels.')
print()
print('RESULTS:')
print(f'  Gemma-2B accuracy (reference) : 66.3%')
print(f'  Phi-3-mini accuracy (this run) : {prm_accuracy:.1%}')

diff = prm_accuracy - 0.663
direction = f'+{diff:.1%}' if diff >= 0 else f'{diff:.1%}'
print(f'  Difference vs Gemma-2B        : {direction}')
print()

if prm_accuracy > 0.80:
    verdict = '✅  GO'
    msg = 'Phi-3 hidden states strongly encode step quality.\n  The PRM pipeline is viable with this model.'
elif prm_accuracy > 0.70:
    verdict = '⚠️  WEAK'
    msg = 'Some signal present but not strong enough.\n  Try more data or a dedicated verifier model.'
else:
    verdict = '❌  NO-GO'
    msg = 'No meaningful signal. Hidden states are not\n  sufficient to classify PRM800K steps reliably.'

print(f'  Verdict : {verdict}')
print(f'  Reason  : {msg}')
print()
print('THRESHOLDS:')
print('  > 80%   → GO   — strong signal')
print('  70-80%  → WEAK — partial signal')
print('  < 70%   → NO-GO (Gemma-2B was here at 66.3%)')
print()
print('CACHE FILES (safe to delete to reset):')
print('  /content/phi3_prm_embeddings.npz — Phi-3 step embeddings')
print('=' * 60)